# Run 001 — Qwen2-VL-7B-Instruct, 8-bit QLoRA, 10k samples

| | |
|---|---|
| **run_id** | `run001_qwen2vl_8bit_10k` |
| **Model** | `Qwen/Qwen2-VL-7B-Instruct` |
| **Quantisation** | 8-bit — flip `quantization.bits` to `4` if this OOMs |
| **Adapter** | LoRA `r=16`, `alpha=32` on `q_proj,k_proj,v_proj,o_proj` of the **language model only** (vision tower frozen) |
| **Visual tokens** | `max_pixels = 256 x 28 x 28` — *the* memory knob |
| **Batch** | 1 x 8 accumulation = effective 8 |
| **Schedule** | 3 epochs, cosine, `warmup_ratio=0.03`, `lr=2e-4`, `paged_adamw_8bit`, fp16 |
| **Loss** | completion tokens only, **class-weighted** (`sqrt_inverse`) |
| **Train / Eval** | 10,000 stratified rows / **frozen 5,000-row eval split** |

### What this run tests

Whether an 8-bit QLoRA fine-tune of a 7B VLM on 10k samples learns the **exact
output format** the metric demands, and how much of the remaining gap
post-processing closes. F1 is reported **raw and post-processed**, and **micro
and macro**, so both the value of normalisation and the effect of class
weighting are visible rather than assumed.

### Expected wall-clock (T4)

~7–9 hours: ~5.5–7.5 h training (3,750 optimiser steps) + ~35–50 min generation
over 5,000 eval rows + ~20 min image download. Inside Kaggle's 12-hour limit,
but not by much — **use T4 x2, not P100**.

> **Run the smoke test in section 7 first.** It exercises the identical path on
> 64 rows in ~10–20 minutes. Finding a bug at hour 8 wastes the session.

## 1. Setup — locate the repo

In [ ]:
# Locate the repo and put its `src/` on sys.path.
#
# Layouts supported, in priority order:
#   1. repo git-cloned into the session -> /kaggle/working/Amazon-ML-24/src
#   2. repo uploaded as a Kaggle Dataset -> /kaggle/input/<slug>/.../src
#   3. running locally from the repo itself
#
# No dataset slug is hardcoded. paths.py finds the competition CSVs wherever
# they are mounted, including the official nested layout
# "<slug>/student_resource 3/dataset/".
import os, sys, glob, subprocess
from pathlib import Path

# Must happen before ANY import that touches CUDA (torch, bitsandbytes). On
# T4 x2, a second visible GPU makes Trainer double the batch size -> OOM.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_URL = "https://github.com/MurtuzaShaikh26/Amazon-ML-24.git"
CLONE_DIR = Path("/kaggle/working/Amazon-ML-24")


def find_src():
    candidates = [CLONE_DIR / "src"]
    for root in sorted(glob.glob("/kaggle/input/*/")):
        candidates.append(Path(root) / "src")
        candidates.extend(Path(p) for p in glob.glob(root + "*/src"))
    candidates.extend(Path(p) for p in sorted(glob.glob("/kaggle/working/*/src")))
    candidates.extend([Path.cwd() / "src", Path.cwd().parent / "src"])
    for c in candidates:
        if (c / "amlc24" / "__init__.py").exists():
            return c.resolve()
    return None


# Always run the latest GitHub code. A Kaggle Dataset linked to GitHub never
# syncs on push, so an attached repo dataset is a stale snapshot; the fresh
# clone must win. The dataset copy is only a fallback when there is no internet.
if CLONE_DIR.exists():
    print("Existing clone found; pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "fetch", "--depth", "1", "origin", "main"],
                   capture_output=True, text=True)
    subprocess.run(["git", "-C", str(CLONE_DIR), "reset", "--hard", "origin/main"],
                   capture_output=True, text=True)
else:
    print("Cloning repo...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
                   capture_output=True, text=True)

if (CLONE_DIR / "src" / "amlc24" / "__init__.py").exists():
    SRC = (CLONE_DIR / "src").resolve()
    commit = subprocess.run(["git", "-C", str(CLONE_DIR), "log", "-1", "--format=%h %s"],
                            capture_output=True, text=True).stdout.strip()
    print("Running GitHub commit:", commit)
else:
    SRC = find_src()
    print("WARNING: clone failed (internet off?). Using fallback copy:", SRC)
assert SRC is not None, "Could not find the amlc24 package; check the clone above."
sys.path.insert(0, str(SRC))

# Drop any already-imported copy so a pull takes effect without a kernel restart.
for name in [m for m in list(sys.modules) if m == "amlc24" or m.startswith("amlc24.")]:
    del sys.modules[name]

import amlc24
from amlc24.logging_utils import setup_logging
from amlc24.paths import describe

setup_logging()
print()
print("amlc24", amlc24.__version__, " src:", SRC)

info = describe()
for k, v in info.items():
    print(f"  {k:20s} {v}")

if not info.get("train_csv_found"):
    print()
    print("*** train.csv NOT FOUND ***")
    print("Attach the competition dataset: '+ Add Input' in the notebook sidebar,")
    print("search for the Amazon ML Challenge 2024 dataset, add it, then re-run.")
    print("Currently mounted inputs:", info.get("mounted_inputs"))
    raise SystemExit("Competition dataset not attached.")

print()
print("Data located. Ready.")

## 2. Guarded installs

Kaggle's base image already carries most of this. Only what is missing or too
old gets installed.

In [ ]:
import importlib, importlib.metadata, subprocess, sys

REQUIRED = {
    "transformers": "4.45.0",
    "peft": "0.13.0",
    "bitsandbytes": "0.44.0",
    "accelerate": "0.34.0",
    "trl": "0.11.0",
    "qwen_vl_utils": None,
}
PIP_NAME = {"qwen_vl_utils": "qwen-vl-utils"}


def installed_version(mod):
    try:
        return importlib.metadata.version(PIP_NAME.get(mod, mod))
    except Exception:
        return None


missing = []
for mod, min_version in REQUIRED.items():
    have = installed_version(mod)
    if have is None:
        missing.append(PIP_NAME.get(mod, mod))
    elif min_version:
        current = tuple(int(x) for x in have.split(".")[:3] if x.isdigit())
        wanted = tuple(int(x) for x in min_version.split("."))
        if current < wanted:
            missing.append(f"{PIP_NAME.get(mod, mod)}>={min_version}")

if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *missing],
                   check=False)
else:
    print("All requirements already satisfied.")

In [ ]:
import importlib

for mod in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "trl"]:
    try:
        m = importlib.import_module(mod)
        print(f"{mod:16s} {getattr(m, '__version__', '?')}")
    except ImportError as exc:
        print(f"{mod:16s} MISSING ({exc})")

## 3. GPU check

**T4 is required.** T4 is Turing (SM 7.5): no bf16, no flash-attention-2, which
is why the config uses fp16 + `sdpa`.

Kaggle's default accelerator is **T4 x2**, so torch sees two devices and HF
`Trainer` would wrap the model in `DataParallel` — which breaks an 8-bit model
already pinned to GPU 0. `pin_single_gpu()` prevents that and must run before
torch initialises CUDA.

In [ ]:
from amlc24.models.qwen2vl import pin_single_gpu

pin_single_gpu(0)

from amlc24.models.qwen2vl import gpu_report

report = gpu_report()
for k, v in report.items():
    print(f"  {k:22s} {v}")

if not report.get("cuda"):
    raise SystemExit("No GPU. Settings -> Accelerator -> 'GPU T4 x2', then restart.")

if report.get("is_p100"):
    print()
    print("!" * 76)
    print("!!  WARNING: P100 detected, not T4.")
    print("!!  P100 is materially slower for this fp16 workload and this run")
    print("!!  may exceed the 12-hour Kaggle session limit.")
    print("!!  Switch to 'GPU T4 x2' and restart the session.")
    print("!" * 76)
elif report.get("is_t4"):
    print()
    print("T4 confirmed -- fp16 + sdpa, as configured.")

if report.get("device_count", 1) > 1:
    print()
    print("NOTE: more than one GPU still visible, so pin_single_gpu() ran after")
    print("CUDA was initialised. Restart the kernel and run from the top,")
    print("otherwise Trainer will use DataParallel.")

if report.get("total_vram_gb", 0) < 15:
    print()
    print("WARNING: less than 15 GB VRAM visible; consider quantization.bits = 4.")

## 4. Config

Loaded from `configs/run001_qwen2vl_8bit_10k.yaml`, which `extends: base.yaml`.
The `config_hash` goes on the leaderboard row so any score traces back to exact
settings.

In [ ]:
from amlc24.config import config_hash, load_config

CONFIG = "run001_qwen2vl_8bit_10k"
cfg = load_config(CONFIG)

print("run_id     ", cfg.run_id)
print("hash       ", config_hash(cfg))
print("description", cfg.description)
print()
print("model      ", cfg.model.id, f"({cfg.quantization.bits}-bit, attn={cfg.model.attn_implementation})")
print("visual toks", cfg.processor.max_pixels_tokens, "max =",
      cfg.processor.max_pixels_tokens * 28 * 28, "px")
print("lora       ", f"r={cfg.lora.r} alpha={cfg.lora.alpha}", list(cfg.lora.target_modules))
print("train      ", f"{cfg.train.num_train_epochs} epochs,",
      f"bs={cfg.train.per_device_train_batch_size} x accum={cfg.train.gradient_accumulation_steps},",
      f"lr={cfg.train.learning_rate}")
print("class wts  ", dict(cfg.train.class_weights))
print("postprocess", f"number_format={cfg.postprocess.number_format},",
      f"range_rule={cfg.postprocess.range_rule}")
print("data       ", f"{cfg.data.train_size} train / {cfg.data.eval_size} eval (FROZEN)")

## 5. Images

The pipeline downloads only the images the frozen split references, resized so
the longest side is 448px (matching the `max_pixels` cap) as JPEG q90. Existing
files are skipped, so this is resumable.

**If you have uploaded the resized images as a private Kaggle Dataset**,
`paths.py` finds it automatically and this becomes a no-op — the recommended
workflow, since sessions are ephemeral and re-downloading wastes ~20 minutes of
the 12-hour budget every run.

In [ ]:
from amlc24.paths import IMAGE_DIR

print("Image dir:", IMAGE_DIR)
print("Exists:   ", IMAGE_DIR.exists())
if IMAGE_DIR.exists():
    n = sum(1 for _ in IMAGE_DIR.glob("*.jpg"))
    print(f"Cached:    {n:,} jpg files")
    if n > 10000:
        print("-> Pre-uploaded image dataset detected; download will be a no-op.")
    else:
        print("-> Will download the missing images (~20 min for a cold start).")
else:
    print("-> No image cache; the pipeline will download into the working dir.")

## 6. Split creation / verification

The 5,000-row eval split is **frozen**: loaded from
`results/splits/split_seed42.json`, regenerated, and compared. Any disagreement
raises `SplitMismatch` rather than silently overwriting.

In [ ]:
import pandas as pd
from amlc24.data.load import load_train
from amlc24.data.splits import get_or_create_split

train_all = load_train()
split = get_or_create_split(
    train_all,
    seed=int(cfg.seed),
    eval_size=int(cfg.data.eval_size),
    train_size=int(cfg.data.train_size),
    path=cfg.data.split_file,
)

print(f"eval_5k       {len(split['eval_5k']):,} rows  (FROZEN, immutable)")
print(f"train_subset  {len(split.get('train_subset', [])):,} rows")
print(f"pool          {len(split.get('pool', [])):,} rows")
print(f"overlap       {len(set(split['eval_5k']) & set(split.get('train_subset', [])))}")
print(f"strata        {split['stratify']['n_strata']}  key = {split['stratify']['key']}")

table = split.get("proportion_table")
if table is not None:
    display(pd.DataFrame(table) if not hasattr(table, "columns") else table)

### Class weights

The EDA measured a **31.5x** imbalance across `entity_name` (`item_weight`
38.95% vs `maximum_weight_recommendation` 1.24%). Unweighted, ~72% of the
gradient comes from weight-and-dimension rows.

`sqrt_inverse` caps the spread near 5.6x. Full `inverse` would give the rarest
class a 31x multiplier — at batch size 1 that lands on a whole step's gradient,
an fp16 loss-spike risk.

In [ ]:
from amlc24.data.load import load_split_frames
from amlc24.train.weighting import weights_from_config

_eval_df, _train_df = load_split_frames(split, train_all)
class_weights = weights_from_config(
    _train_df["entity_name"].tolist(), cfg.get("train", {}).get("class_weights", {})
)

if class_weights:
    counts = _train_df["entity_name"].value_counts()
    display(pd.DataFrame({
        "entity_name": list(class_weights),
        "n_train": [int(counts.get(e, 0)) for e in class_weights],
        "loss_weight": [round(w, 3) for w in class_weights.values()],
    }).sort_values("n_train", ascending=False).reset_index(drop=True))
    spread = max(class_weights.values()) / min(class_weights.values())
    print(f"spread (max/min) = {spread:.2f}x")
else:
    print("Class weighting disabled -- using the model's own unweighted loss.")

## 7. Train

One call: downloads images, builds datasets with completion-only label masking,
loads the quantised model, attaches LoRA, trains, generates over the frozen eval
5k, scores raw **and** post-processed, and writes every artefact under
`results/runs/{run_id}/`.

> **Smoke test first.** The cell defaults to `SMOKE = True`, exercising the
> identical path on 64 rows in ~10–20 minutes. A full run is ~7–9 hours, and
> finding a collator or OOM bug at hour 8 wastes the session. Once it passes,
> set `SMOKE = False` and re-run.

In [ ]:
from amlc24.pipeline.run_finetune import run_finetune

# A smoke run exercises the identical code path -- image download, collator,
# label masking, 8-bit quantisation, LoRA attach, a training step, generation,
# post-processing, scoring, artefact writing -- on 64 train / 64 eval rows.
SMOKE = True

if SMOKE:
    print("SMOKE TEST: 64 train / 64 eval rows. Scores are NOT meaningful.")
    print()
    result = run_finetune(CONFIG, max_train_rows=64, max_eval_rows=64)
    print()
    print("=" * 70)
    print("SMOKE TEST PASSED -- the full path works end to end.")
    print("Set SMOKE = False and re-run this cell for the real run.")
    print("The scores above come from 64 rows; ignore them.")
    print("=" * 70)
else:
    result = run_finetune(CONFIG)

metrics = result["metrics"]
print()
print(f"  F1 raw            {metrics['raw']['f1']:.4f}")
print(f"  F1 post-processed {metrics['post']['f1']:.4f}")
print(f"  macro F1 post     {metrics['macro_f1_post']:.4f}")

## 8. Results

### 8.1 Loss curves

In [ ]:
from amlc24.train.trainer import plot_loss_curves

history = result["history"]
if len(history):
    display(history.tail(10))
    plot_loss_curves(history);
else:
    print("No loss history recorded.")

### 8.2 Overall F1 — raw vs post-processed, micro vs macro

The raw/post delta is the measured value of `postprocess/`. Micro F1 is
dominated by `item_weight` (38.95% of the data); **macro F1 weights all eight
entities equally, so it is the number that moves when class weighting helps a
rare entity.**

In [ ]:
raw, post = result["raw"], result["post"]
summary = pd.DataFrame([
    {"variant": "raw generation", **{k: raw[k] for k in
     ["f1", "precision", "recall", "tp", "fp", "fn", "tn"]}},
    {"variant": "post-processed", **{k: post[k] for k in
     ["f1", "precision", "recall", "tp", "fp", "fn", "tn"]}},
])
display(summary.style.format({"f1": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}"}))

print(f"Post-processing changed micro F1 by {post['f1'] - raw['f1']:+.4f}")
print()
print("MICRO vs MACRO")
print(f"  micro (overall)     raw={raw['f1']:.4f}  post={post['f1']:.4f}")
print(f"  macro (per-entity)  raw={result['macro_f1_raw']:.4f}  post={result['macro_f1_post']:.4f}")
print()
print("Post-processing rules fired:")
for rule, count in sorted(metrics["postprocess_rules"].items(), key=lambda kv: -kv[1])[:15]:
    print(f"  {rule:40s} {count:,}")

### 8.3 F1 by entity — the class-wise evaluation

`f1_delta_from_postprocess` shows which entities normalisation rescued.

In [ ]:
cols = ["f1", "precision", "recall", "f1_raw", "f1_delta_from_postprocess", "accuracy"]
display(result["by_entity"].style.format({c: "{:.4f}" for c in cols}))

### 8.4 F1 by ground-truth unit

Grouping by the *true* unit separates unit-formatting failures from value
failures — two different problems with two different fixes.

In [ ]:
cols = ["f1", "precision", "recall", "accuracy"]
display(result["by_unit"].head(30).style.format({c: "{:.4f}" for c in cols}))

### 8.5 F1 by `group_id` — the category-wise breakdown

750 product categories with a long tail, so groups under 20 eval rows are
pooled. A category far below the rest usually means a product type whose
packaging the model cannot read — a data problem, not a formatting one.

In [ ]:
by_group = result.get("by_group")
if by_group is not None:
    cols = ["f1", "precision", "recall", "accuracy"]
    display(by_group.head(25).style.format({c: "{:.4f}" for c in cols}))
else:
    print("No group_id column in the predictions frame.")

### 8.6 Error analysis — where the next rules come from

The most frequent `(predicted, actual)` mismatch pairs per entity. A repeated
pair with `same_number_diff_unit = True` is a missing alias and is free score;
scattered numeric errors are a model-capacity problem instead.

In [ ]:
errors = result["errors"]
recoverable = errors[errors["same_number_diff_unit"]]
print(f"{len(errors)} distinct mismatch pairs; {len(recoverable)} differ only in the unit")
display(errors.head(40))

In [ ]:
preds = result["predictions"]
display(preds[preds["y_true"] != preds["y_pred_post"]].head(25))

### 8.7 Leaderboard

In [ ]:
from amlc24.results.tracker import read_leaderboard

display(read_leaderboard())

## 9. Generate the competition submission

Runs the fine-tuned model over `test.csv` (131,187 rows, **no labels**) and
writes `index,prediction`.

`write_submission` enforces the official rules plus one the shipped `sanity.py`
does not: **row count must equal `test.csv` exactly**, or the file is not
evaluated. Missing indices are filled with an empty prediction — one false
negative each, versus a discarded submission.

> Slow (~131k rows, several hours). Skip while iterating; run only to submit.

In [ ]:
GENERATE_SUBMISSION = False   # flip to True when you actually want to submit

if GENERATE_SUBMISSION:
    from amlc24.pipeline.run_finetune import run_test_predictions

    out = run_test_predictions(CONFIG)
    print("Wrote", out["path"])
    display(out["submission"].head(10))

    n_empty = (out["submission"]["prediction"] == "").sum()
    print(f"{len(out['submission']):,} rows, {n_empty:,} empty predictions")
else:
    print("Skipped. Set GENERATE_SUBMISSION = True to produce submission.csv.")

## 10. Package the run for download

Zips `results/runs/{run_id}/` (metrics, predictions, breakdowns, logs —
excluding adapter binaries). The LoRA adapter is saved separately under
`checkpoints/adapter_best/` (~40 MB).

In [ ]:
tracker = result["tracker"]
zip_path = tracker.zip_artifacts()

print("Run artefacts:", tracker.dir)
for p in sorted(tracker.dir.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(tracker.dir)}  ({p.stat().st_size / 1024:.1f} KB)")
print()
print("Zipped to:", zip_path)

## 11. Observations

<!-- Fill this in after the run. Record what you learned in NOTES.md too. -->